In [2]:
# pip install sounddevice soundfile groq langchain-groq
from dotenv import load_dotenv
load_dotenv()

import sounddevice as sd
import soundfile as sf
from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

client = Groq()
model = ChatGroq(model="openai/gpt-oss-120b")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment analysis system. Classify the user's text "
               "as exactly one word: Positive, Negative, or Neutral. Reply with that word only."),
    ("user", "{text}"),
])
sentiment_chain = prompt | model | StrOutputParser()

def record(seconds=5, path="mic.wav", rate=16000):
    input(f"Press Enter, then talk for {seconds} seconds...")
    audio = sd.rec(int(seconds * rate), samplerate=rate, channels=1)
    sd.wait()
    sf.write(path, audio, rate)
    return path

def transcribe(path):
    with open(path, "rb") as f:
        return client.audio.transcriptions.create(
            file=f, model="whisper-large-v3", language="ar"
        ).text

while True:
    text = transcribe(record(5))
    print("You said:", text)
    print("Sentiment:", sentiment_chain.invoke({"text": text}))
    if input("Again? (y/n) ") != "y":
        break

You said:  مرحباً، اسمي محمد، أنا من جوردان وأنا مصنع مفرد.
Sentiment: Neutral
You said:  مرحباً كيف حالك؟
Sentiment: Neutral
You said:  شكرا
Sentiment: Positive
You said:  مرحباً، اسمي محمد، أنا أعيش في جورجين
Sentiment: Neutral
